# 07 · Tracing and eval

To compare three frameworks, every run has to be measured the same way: the same definition of a token,
an LLM call, a tool call and a second. This notebook connects CrewAI to the shared harness in
`agent_contracts.harness`, measures runs of the research flow from notebook 05, and scores them with the
shared scoring, which is how the capstone's `run_eval.py` will work.

**You will learn**

- what the shared harness measures, and where each number comes from
- how a CrewAI event listener feeds model calls into it
- how to run a small, scored evaluation and write it in the shared results format

In [1]:
from crew_agents.metrics import harness_listener  # import crew_agents first: telemetry off
from crew_agents.research_flow import ResearchFlow, to_report

from agent_contracts.harness import RunMetrics, build_results, verify_model
from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report

verify_model()  # stops with the command that fixes it if Ollama or the shared model is missing
questions = {q["id"]: q for q in load_questions()}  # the shared questions by id, each with its expected facts

## The shared definitions

| Number | Definition | Where it comes from |
|---|---|---|
| LLM calls | every model call in the run | the framework's callbacks, events or hooks |
| input and output tokens | summed over every model call | the counts Ollama reports with each response |
| tool calls | every attempted call, refused ones included | the shared tools record them (`research.tool_calls()`) |
| latency | wall clock for the whole question | `RunMetrics.measure()` |

Tool calls are counted inside the shared tools, so no framework can count them differently. Model calls
have to come from the framework, because only it sees them; in CrewAI that is the **event bus**.

`HarnessListener`, from `crew_agents/metrics.py`, is the whole bridge between CrewAI's event bus and the shared `RunMetrics`:

```python
class HarnessListener(BaseEventListener):
    def __init__(self) -> None:
        self.metrics: RunMetrics | None = None  # None: not measuring right now, so events are ignored
        super().__init__()  # BaseEventListener's __init__ calls setup_listeners below

    def setup_listeners(self, crewai_event_bus) -> None:
        # Subscribe to one event type: CrewAI publishes LLMCallCompletedEvent after every model call.
        @crewai_event_bus.on(LLMCallCompletedEvent)
        def on_llm_call(source, event: LLMCallCompletedEvent) -> None:
            if self.metrics is not None:
                self.metrics.add_llm_call(*_usage(event.usage))  # (input tokens, output tokens)

    # Used as `with listener.measuring(metrics):`. Only calls made inside the block are counted.
    @contextmanager
    def measuring(self, metrics: RunMetrics):
        """Count every model call made inside the `with` block into `metrics`."""
        self.metrics = metrics
        try:
            yield metrics  # the body of the `with` block runs here
        finally:
            crewai_event_bus.flush()  # events are delivered asynchronously; wait for the last ones
            self.metrics = None
```

CrewAI publishes an `LLMCallCompletedEvent` after every model call, wherever it comes from: an agent,
a crew, a flow step, or a direct `LLM.call`. Events are delivered **asynchronously**, so `measuring` flushes
the bus before it stops counting; without that, the last calls of a run can arrive too late and be missed.

## One measured run

In [3]:
async def run_question(qid: str) -> dict:
    """Run one question through the flow, measured and scored the shared way."""
    q = questions[qid]
    research.reset()  # clear the shared tools' call record, so only this question's calls are counted
    metrics = RunMetrics()
    flow = ResearchFlow()  # a fresh flow per question, so no state carries over
    with metrics.measure(), harness_listener.measuring(metrics):  # time the run, and count every model call
        await flow.kickoff_async(inputs={"question_id": qid, "question": q["question"]})
    report = to_report(flow.state)
    scored = score_report(report, q)  # compares the report with the question's expected facts
    # One result per question, in the shared format.
    return {"id": qid, "passed": scored["passed"], "scores": scored["scores"], "problems": scored["problems"],
            "report": report.model_dump(), "tool_calls": research.tool_calls(),
            "budget_exhausted": research.budget_exhausted(), **metrics.as_dict(), "error": None}

Now run one question, `Q07_deployment`, and print what came back: whether it passed, what the run cost,
and the answer with its scores.

In [4]:
item = await run_question("Q07_deployment")

print("passed:         ", item["passed"])
print("model calls:    ", item["llm_calls"])
print("tokens in / out:", item["input_tokens"], "/", item["output_tokens"])
print("time:           ", f"{item['latency_s']:.0f} s")
print("tools called:   ", ", ".join(item["tool_calls"]) or "none")
print("answer:         ", item["report"]["answer"])
print("fact recall:    ", item["scores"]["fact_recall"], "(share of the expected facts the answer contains)")
print("facts missing:  ", ", ".join(item["scores"]["facts_missing"]) or "none")
print("no stale figure:", item["scores"]["contradiction_free"])
print("citations valid:", item["scores"]["citations_valid"])
print("problems:       ", "; ".join(item["problems"]) or "none")

passed:          True
model calls:     10
tokens in / out: 17595 / 579
time:            242 s
tools called:    web_search, web_search, web_search, web_search, web_search, web_search
answer:          Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces, according to the 2025 annual report [F04].
fact recall:     1.0 (share of the expected facts the answer contains)
facts missing:   none
no stale figure: True
citations valid: True
problems:        none


Q07 asks how many units Veltara has deployed and in how many provinces. The 2023 article still says
9 provinces; the current figure is 14. Mentioning 9 at all fails `contradiction_free`.

## A small eval

An evaluation is the same thing over a fixed set of cases, scored automatically, so you can re-run it after
every change and see whether things got better. These are the same questions the LangGraph tutorial uses:
a calculation (Q03), a recall with a distractor article about another company (Q08), and the Series B
round (Q04). Each takes a minute or two on a 16 GB M4 Mac.

In [5]:
# Run the other three questions with the same function and keep every result in one list.
items = [item]
for qid in ("Q03_revenue_growth", "Q08_recall", "Q04_series_b"):
    items.append(await run_question(qid))
    print("finished", qid)

finished Q03_revenue_growth


finished Q08_recall


finished Q04_series_b


When the runs finish, print one row per question: whether it passed, how much of the expected facts it
recalled, and what it cost.

In [6]:
# One row per question: pass or fail, recall of the expected facts, and the cost of the run.
print(f"{'question':20} {'pass':5} {'recall':6} {'calls':5} {'tokens':>7} {'secs':>5}  tools")
for it in items:
    print(f"{it['id']:20} {str(it['passed']):5} {it['scores']['fact_recall']:6} {it['llm_calls']:5} "
          f"{it['input_tokens'] + it['output_tokens']:7} {it['latency_s']:5.0f}  {', '.join(it['tool_calls'])}")

passed = sum(it["passed"] for it in items)
print(f"\n{passed} of {len(items)} questions passed")

# For any question that failed a check, show what the check found and the answer that triggered it.
for it in items:
    if it["problems"]:
        print(f"\n{it['id']} failed a check")
        print("  problems:", "; ".join(it["problems"]))
        print("  answer:  ", it["report"]["answer"])

question             pass  recall calls  tokens  secs  tools
Q07_deployment       True     1.0    10   18174   242  web_search, web_search, web_search, web_search, web_search, web_search
Q03_revenue_growth   True     1.0     7    7350   156  web_search, web_search, calculator
Q08_recall           True     1.0     6    5339    98  web_search, web_search
Q04_series_b         True     1.0     5    2986    84  web_search

4 of 4 questions passed


When we ran it, all four questions passed. The LangGraph tutorial ran the same four questions with the
same instructions and model, and they all passed there too. The cost differed:

| | LangGraph (our run) | CrewAI (our run) |
|---|---|---|
| passed | 4 of 4 | 4 of 4 |
| total tokens | 15,669 | 33,849 |
| total time | 439 s | 580 s |
| web searches | 5 | 11 |

CrewAI's research agent searched more often (six searches on Q07, against one), and each CrewAI agent
call carries a longer prompt, built from the role, goal and backstory. These are single runs of four
questions on one laptop, so treat the gap as a first signal, not a verdict; the capstone evaluations run
every question in every framework.

## The shared results format

`build_results` wraps the items in the format every framework writes, and computes the summary the same way
for all of them. The capstone's `run_eval.py` writes it to `results/research/crewai.json`; this notebook
only prints it, so a tutorial run never overwrites real results.

In [7]:
from importlib.metadata import version

from agent_contracts.harness import load_settings

# Wrap the four results in the shared format; the framework's name and version are recorded in the file.
results = build_results("research", "crewai", version("crewai"), load_settings().model, items)

summary = results["summary"]  # computed the same way for every framework
print("model:        ", results["model"])
print("passed:       ", f"{summary['passed']} of {summary['total']} questions")
print("total tokens: ", summary["total_tokens"])
print("total time:   ", f"{summary['total_latency_s']:.0f} s")
print("fields saved for each question:", ", ".join(sorted(results["questions"][0])))

model:         qwen3-agents
passed:        4 of 4 questions
total tokens:  33849
total time:    580 s
fields saved for each question: budget_exhausted, error, id, input_tokens, latency_s, llm_calls, output_tokens, passed, problems, report, scores, tool_calls


## Seeing inside a run

Numbers say *how much*; traces say *what happened*. In CrewAI, the simplest trace is the console:
`verbose=True` on an agent or crew, or the flow panels (notebooks 03 and 05). The event bus is the
general mechanism: every agent step, tool use, task and flow method emits events that a listener like the
one above can record. CrewAI also offers hosted tracing (`CREWAI_TRACING_ENABLED`), which sends traces to
CrewAI's cloud and is switched off in this local-only tutorial.

## How the others do this

- **LangGraph** (LangChain) passes a callback handler in the run's `config`; its `on_llm_end` sees every
  model call, including those of an agent nested inside a graph node. Callbacks run in line with the
  call, so there is nothing to flush.
- **Strands** uses hooks: `AfterModelCallEvent` and `AfterToolCallEvent` fire around each call, and an
  agent's result carries `metrics.accumulated_usage`.

All three feed the same `RunMetrics`, and all three count tool calls through the shared tools.

## Takeaways

- Fix the definitions once, in shared code, then connect each framework to them.
- Count what you can outside the framework (tool calls); take only model calls from the framework.
- CrewAI's events arrive asynchronously: flush before you read the numbers.
- An eval is a fixed set of cases, scored automatically, re-run after every change.
- That completes the components. The capstones build ShipFast and the research agent in full.